In [ ]:
MODEL_ID = "usecase2"
# October 30 sample; replace with the raw training CSV from your run.
RAW_DATA_URI = "s3://src1007cced-243788878595-us-east-1/wsf-mlops/src1007cced-i1030a/models/usecase2/runs/cd97jkwzjyqo/data/raw/data.csv"
MODEL_METADATA_URI = "none"  # Or the preliminary model_metadata.json S3 URI.


In [ ]:
from pathlib import Path
from tempfile import mkdtemp
from urllib.parse import urlsplit
import importlib
import json
import os
import sys

import boto3
import pandas as pd
from botocore.config import Config as ClientConfig
from IPython import get_ipython
from IPython.display import display

get_ipython().run_line_magic("load_ext", "autoreload")
get_ipython().run_line_magic("autoreload", "2")

REPOSITORY = next(p for p in (Path.cwd(), *Path.cwd().parents)
                  if (p / "mlops-models/src/mlops_models").is_dir())
sys.path.insert(0, str(REPOSITORY / "mlops-models/src"))
parameter_path = REPOSITORY / "mlops-models/src/mlops_models/models" / MODEL_ID / "parameters.json"

work_directory = Path(mkdtemp(prefix="feature-selection-"))
data_directory = work_directory / "raw"
output_root = work_directory / "output"
data_directory.mkdir()
output_root.mkdir()

os.environ.update({
    "MODEL_ID": MODEL_ID,
    "MLOPS_MODEL_METADATA_URI": MODEL_METADATA_URI.strip(),
    "MLOPS_TRACKING_URI": "",
    "MLOPS_WRITE_RUN_REPORT": "0",
    "MLOPS_LOG_ARTIFACTS": "0",
})

s3 = boto3.client("s3", config=ClientConfig(
    connect_timeout=5, read_timeout=15, retries={"total_max_attempts": 2}))
address = urlsplit(RAW_DATA_URI)
if address.scheme != "s3" or not address.netloc or not address.path.strip("/"):
    raise ValueError("Set RAW_DATA_URI to an S3 CSV object path in the first cell.")
print(f"Downloading {RAW_DATA_URI} -> {data_directory / 'data.csv'}", flush=True)
s3.download_file(address.netloc, address.path.lstrip("/"), str(data_directory / "data.csv"))
print("Parameters used by the entrypoint:", parameter_path)


In [ ]:
# Optional inspection; execution does not depend on this cell.
display(json.loads(parameter_path.read_text()))
display(pd.read_csv(data_directory / "data.csv", nrows=10, dtype=str))


In [ ]:
# Reference only: copy into your feature-selection metadata branch.
# import json
# import os
# import boto3
# from urllib.parse import urlsplit
#
# uri = os.getenv("MLOPS_MODEL_METADATA_URI", "").strip()
# if uri and uri.lower() != "none":
#     address = urlsplit(uri)
#     with boto3.client("s3").get_object(
#         Bucket=address.netloc,
#         Key=address.path.lstrip("/"),
#     )["Body"] as body:
#         metadata = json.load(body)
#
#     features = list(metadata["feature_mapping"])


In [ ]:
# A fresh output folder keeps plots from an earlier fit out of a metadata-only run.
output_directory = Path(mkdtemp(prefix="run-", dir=output_root))
os.environ.update({
    "MLOPS_MODEL_METADATA_URI": MODEL_METADATA_URI.strip(),
    "SM_OUTPUT_DATA_DIR": str(output_directory),
    "SM_OUTPUT_DIR": str(output_directory),
})

from mlops_models.entrypoints import feature_selection

feature_selection = importlib.reload(feature_selection)
print("Entrypoint:", feature_selection.__file__)
print("Metadata input:", os.environ["MLOPS_MODEL_METADATA_URI"])
result_directory = feature_selection.execute(
    model_id=MODEL_ID,
    input_path=data_directory,
    output_dir=output_directory,
)
print("Output directory:", output_directory)
result_directory
